In [ ]:
# Conv2d двумерный сверточный слой
# MaxPool2d пулинг, понижает размерность карты признаков

In [ ]:
import json
import os
import torch

import torch.nn as nn
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torch.optim as optim

from tqdm import tqdm
from PIL import Image


class SunDataset(data.Dataset):
    def __int__(self, path, train=True, transform=None):
        self.path = os.path.join(path, 'train' if train else 'test')
        self.transform = transform

        with open(os.path.join(self.path, 'format.json'), 'r') as fp:
            self.format = json.load(fp)
        
        self.length = len(self.format)
        self.files = tuple(self.format.keys())
        self.targets = tuple(self.format.values())

        def __getitem__(self, item):
            path_file = os.path.join(self.path, self.files[item])
            img = Image.open(path_file).convert('RGB')

            if self.transform:
                img = self.transform(img)
            
            return img, torch.tensor(self.targets[item], dtype=torch.float32)
        
        def __len__(self):
            return self.length


transforms = tfs.Compose([tfs.ToImage(), tfs.ToDtype(torch.float32, scale=True)])
d_train = SunDataset('datasets/cnn', transform=transforms)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)


model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding='same'),  # Размерность тензора (batch, 3, 256, 256), 32 фильтра размером 3x3
    nn.ReLU(),                            # Функция активации
    nn.MaxPool2d(2),                      # Линейные размеры карты признаков уменьшается в 2 раза
    nn.Conv2d(32, 8, 3, padding='same'),  # Размерность тензора (batch, 32, 128, 128), 8 фильтров размером 3x3
    nn.ReLU(),                            # Функция активации
    nn.MaxPool2d(2),                      # Линейные размеры карты признаков уменьшается в 2 раза
    nn.Conv2d(8, 4, 3, padding='same'),   # Размерность тензора (batch, 8, 64, 64), 4 фильтра размером 3x3
    nn.ReLU(),                            # Функция активации
    nn.MaxPool2d(2),                      # Линейные размеры карты признаков уменьшается в 2 раза
    nn.Flatten(),                         # Вытягиваем тензор (batch, 4, 32, 32) в вектор длиной 4096 
    nn.Linear(4096, 128),                 # Полносвязный слой со входом 4096 и скрытый слой 128
    nn.ReLU(),                            # Функция активации
    nn.Linear(128, 2),                    # Полносвязный слой со входом 128 и выходом 2, координаты x и y
)


optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=0.001)
loss_function = nn.MSELoss()


epochs = 5
model.train()


for _e in range(epochs):
    loss_mean = 0
    lm_count = 0
    train_tqdm = tqdm(train_data, leave=True)

    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epochs [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}')


st = model.state_dict()
torch.save(st, 'model_sun.tar')

d_test = SunDataset('datasets/cnn', train=False, transform=transforms)
test_data = data.DataLoader(d_test, batch_size=50, shuffle=False)

Q = 0
count = 0
model.eval()

test_tqdm = tqdm(test_data, leave=True)
for x_test, y_test in test_tqdm:
    with torch.no_grad():
        p = model(x_test)
        Q += loss_function(p, y_test).item()
        count += 1
Q /= count
print(Q)

